In [23]:
!pip install pandas numpy scikit-learn python-dateutil joblib fastapi uvicorn python-multipart openpyxl requests --break-system-packages


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
os.chdir(r"/home/natooa/AIFinance/ml")  # укажите свой реальный путь к папке python

In [2]:
import subprocess
import sys

subprocess.run([sys.executable, "1_generate_data.py"], check=True)
subprocess.run([sys.executable, "2_financial_health.py"], check=True)
subprocess.run([sys.executable, "3_anomaly_detection.py"], check=True)
subprocess.run([sys.executable, "4_forecast.py"], check=True)
subprocess.run([sys.executable, "8_train_and_save_models.py"], check=True)

print("ML pipeline успешно завершён")

/home/natooa/AIFinance/ml/1_generate_data.py:1: SyntaxWarning: invalid escape sequence '\c'
  """


Готово: 9 компаний, 108 финансовых записей, 886 транзакций. Файлы сохранены в /home/natooa/AIFinance/ml/../data/
     company_id  ...                              top_factors_financial
83            7  ...  [debt_to_ebitda=100, current_ratio=79, profit_...
95            8  ...  [debt_to_ebitda=100, current_ratio=79, profit_...
47            4  ...                            [receivables_growth=83]
23            2  ...                                                 []
71            6  ...                                                 []
35            3  ...                                                 []
59            5  ...                                 [current_ratio=51]
11            1  ...                                                 []
107           9  ...  [debt_to_ebitda=100, current_ratio=79, profit_...

[9 rows x 4 columns]
   company_id  anomaly_score                                top_factors_anomaly
0           1           74.2  [сумма транзакции в 5.9x выше истор

In [3]:
import os
print(os.listdir(r"/home/natooa/AIFinance/ml"))

['8_train_and_save_models.py', 'financial_statements.csv', '9_app.py', 'companies.csv', 'financial_health_scores_history.csv', '1_generate_data.py', 'financial_health_scores.csv', '2_financial_health.py', 'settlement_service.ipynb', '7_dashboard.py', '3_anomaly_detection.py', '6_risk_engine.py', 'counterparties.csv', '__pycache__', 'test.csv', 'anomaly_scores.csv', 'transactions.csv', 'bad.csv', '5_pd_model.py', '4_forecast.py']


In [4]:
['1_generate_data.py', '2_financial_health.py', '3_anomaly_detection.py', '4_forecast.py', '5_pd_model.py', '6_risk_engine.py', '7_dashboard.py', '8_train_and_save_models.py', '9_app.py', 'bad.csv', 'server.log', 'server3.log', 'test.csv']

['1_generate_data.py',
 '2_financial_health.py',
 '3_anomaly_detection.py',
 '4_forecast.py',
 '5_pd_model.py',
 '6_risk_engine.py',
 '7_dashboard.py',
 '8_train_and_save_models.py',
 '9_app.py',
 'bad.csv',
 'server.log',
 'server3.log',
 'test.csv']

In [1]:
import threading
import uvicorn
from importlib import import_module

app_module = import_module("9_app")  # загрузит models/*.joblib и подготовит приложение

def run_server():
    uvicorn.run(app_module.app, host="127.0.0.1", port=8001, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(2)
print("Сервер запущен на http://127.0.0.1:8001")

Модели загружены. Функции Stage 1 и Stage 4 импортированы из старых файлов.
Сервер запущен на http://127.0.0.1:8001


In [29]:
import requests, pandas as pd

test_df = pd.DataFrame([{
    "revenue": 460000, "ebitda": 55000, "net_profit": 20000, "debt": 250000, "cash": 15000,
    "current_assets": 180000, "current_liabilities": 200000, "receivables": 90000,
    "payables": 44000, "equity": 100000, "assets": 400000
}])
test_df.to_csv("test.csv", index=False)

response = requests.post(
    "http://127.0.0.1:8001/analyze",
    data={"company_name": "Тестовая компания"},
    files={"file": open("test.csv", "rb")}
)
response.json()

{'company_name': 'Тестовая компания',
 'financial_risk': 'MEDIUM',
 'transaction_anomaly_risk': 'LOW',
 'overall_risk_score': 55.0,
 'risk_level': 'MEDIUM',
 'pd_12m_percent': 98.4,
 'requested_credit_limit': 60000000.0,
 'recommended_credit_limit': 6000000.0,
 'monitoring_level': 'ENHANCED',
 'top_risk_factors': ['current_ratio=79',
  'debt_to_ebitda=71',
  'profit_margin=53',
  'файл транзакций не загружен'],
 'recommended_actions': ['усиленный анализ аналитиком',
  'запросить обновлённую отчётность',
  'мониторинг раз в квартал'],
 'note': 'Forecast (Stage 3) появится после повторной загрузки этой же компании — см. комментарий в коде.'}